# 📓 Semana 3 · Dia 1 — Spark: arquitetura, lazy evaluation e DAG

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (Spark), DEP (performance) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Diagrama mental do pipeline Spark |

---


## 📖 Teoria — A arquitetura do Spark

Um cluster Spark tem:
- **Driver**: 'o maestro'. Recebe seu código, monta o plano de execução e coordena os executores.
- **Executors**: 'os músicos'. Rodam as tarefas (tasks) em paralelo, cada um numa partição dos dados.
- **Partições**: fatias dos dados distribuídas entre os executores — o paralelismo vem daqui.

```
         Driver (plano)
        /      |       \
  Executor   Executor   Executor
  [part 0]   [part 1]   [part 2]
```

Na **Free Edition**, o compute é **serverless**: o Databricks provisiona driver + executores para você, sem configurar máquinas.


## 📖 Teoria — Lazy evaluation e o DAG

O Spark é **preguiçoso (lazy)**: chamar `.filter()`, `.select()`, `.join()` NÃO executa nada — só **constrói um grafo** de transformações, o **DAG (Directed Acyclic Graph)**.

A execução só dispara numa **ação**: `.count()`, `.show()`, `.collect()`, `.write()`, `.saveAsTable()`.

**Consequência prática**: você pode encadear 20 transformações sem custo; o custo vem quando o resultado é materializado. E o Spark **reordena/otimiza** o DAG antes de rodar (otimizador Catalyst + AQE).


## 📖 Teoria — DataFrame vs RDD

O **DataFrame** é a API moderna: colunas tipadas, otimizada pelo Catalyst, com schema. O **RDD** é a API antiga de baixo nível (linhas sem schema).

> 🎯 **Dica de prova (DEA 2026)**: a prova **removeu RDDs** do escopo ('ELT with Spark SQL and Python'). Você NÃO precisa programar RDD — mas entender que por baixo tudo vira tarefas em partições ajuda em entrevistas.


### 💻 Na prática — Primeiro DAG

Crie um DataFrame e observe a preguiça do Spark: as transformações retornam instantaneamente; a ação dispara o trabalho.


In [ ]:
# Transformações (lazy) — retornam na hora
df = spark.range(10_000_000)
transformado = (df
    .withColumn("dobro", df["id"] * 2)
    .filter("dobro % 3 == 0"))
print("Transformação criada (lazy) — nada foi executado ainda.")
print("Tipo:", type(transformado).__name__)

In [ ]:
# Ação (eager) — dispara o trabalho
print("Executando ação count()...")
n = transformado.count()
print("Linhas resultantes:", n)

In [ ]:
# Spark UI: veja o DAG da última ação
# Na UI do notebook, clique no ícone de gráfico (Spark UI) da célula acima.
# Procure a aba SQL/Job: é o DAG (estágios de tasks).
print("O Spark UI mostra o DAG de cada job. Explore depois da execução!")

> 🎯 **Dica de prova**: Pergunta clássica: 'o que dispara a execução no Spark?' → uma **ação**. Transformações são lazy. Outra: 'onde roda a task?' → em um executor, sobre uma partição.


## 🎯 Exercícios de fixação

**1.** Liste 4 ações e 4 transformações.

**2.** Por que `df.filter(...)` não executa nada?

**3.** Qual a diferença entre driver e executor?

**4.** Por que RDDs saíram do escopo da DEA 2026?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Ações e transformações

Ações: count, show, collect, write, saveAsTable, take. Transformações: select, filter, join, groupBy, withColumn, orderBy.

**2.** Lazy

Porque o Spark constrói o DAG (plano) primeiro e só executa quando precisa de um resultado materializado (ação). Isso permite otimizar o plano inteiro.

**3.** Driver vs executor

Driver coordena (planeja, distribui tasks, agrega resultados); executor executa as tasks sobre partições em paralelo.

**4.** RDD fora

A DEA 2026 focou o escopo em ELT com Spark SQL e Python (DataFrame API); RDD é legado, usado raramente em produção.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*